# Beyond Frequency: Knowledge Graphs for Fashion Trend Analysis

**Author:** Diego Félix Beserra de Lima  
**Institution:** UFCG — Programa de Pós-Graduação em Ciência da Computação  
**Discipline:** Reprodutibilidade em Pesquisa em Computação  
**Date:** June 2026

---

## About this Notebook

This notebook implements a comparative experiment between two conditions for answering questions about fashion trends:

- **Condition A (LLM-only):** raw text sent directly to the LLM
- **Condition B (KG+LLM):** structured knowledge graph consulted and sent to the LLM

**Research Question:** To what extent are responses about fashion trends generated via a knowledge graph more interpretable and grounded than responses generated directly from raw text?

**Pipeline:**
```
data/raw/ → [01-clean] → data/processed/
         → [02-extract-triples] → data/processed/triples/
         → [03-build-kg] → data/processed/kg/
         → [04-query] → results/
```

**AI Tool Declaration (CNPq Portaria 2.664/2026):**  
Claude (Anthropic), model claude-sonnet-4-6, was used in June 2026 to assist in script development, pattern analysis of raw data, and pipeline structuring. All content was reviewed and validated by the author, who assumes full responsibility.

---
## Step 0 — Environment Setup

In [ ]:
# Install dependencies
!pip install openai networkx --quiet

In [ ]:
# Clone the repository
!git clone https://github.com/diegofelixlima/beyond-frequency.git
%cd beyond-frequency

In [ ]:
# Load OpenAI API key from Colab Secrets
# To add your key: click the key icon in the left sidebar → Add new secret → Name: OPENAI_API_KEY
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("API key loaded successfully.")

---
## Step 1 — Clean Raw Reviews

Removes site navigation, footer, and metadata from raw Ctrl+A collected texts.  
Keeps only editorial content: title, vibe, review body, quote, and wrap up.

**Input:** `data/raw/*.txt` (immutable — never modified)  
**Output:** `data/processed/*.txt`

In [ ]:
!python src/01-clean.py

---
## Step 2 — Extract Semantic Triples

Uses the OpenAI API (gpt-4o-mini, temperature=0) to extract semantic triples from each review.  
Triples follow the format: `{subject, relation, object}`

**Input:** `data/processed/*.txt`  
**Output:** `data/processed/triples/*.json`

**Model:** gpt-4o-mini  
**Temperature:** 0 (deterministic)

In [ ]:
!python src/02-extract-triples.py

In [ ]:
# Inspect extracted triples from one review
import json

sample_file = "data/processed/triples/2025-lfw-erdem-review.json"
with open(sample_file, "r", encoding="utf-8") as f:
    triples = json.load(f)

print(f"Sample: Erdem — {len(triples)} triples extracted\n")
for t in triples[:10]:
    print(f"  {t['subject']} --[{t['relation']}]--> {t['object']}")
print(f"  ... and {len(triples)-10} more")

---
## Step 3 — Build Knowledge Graph

Constructs a directed graph from all extracted triples using NetworkX.  
Each edge carries the relation type and source review.

**Input:** `data/processed/triples/*.json`  
**Output:** `data/processed/kg/fashion_kg.graphml`

**Note:** NetworkX is used as a simplification for reproducibility.  
The full dissertation implementation will use Neo4j.

In [ ]:
!python src/03-build-kg.py

In [ ]:
# Visualize basic graph statistics
import networkx as nx
import collections

G = nx.read_graphml("data/processed/kg/fashion_kg.graphml")

print(f"Knowledge Graph Statistics")
print(f"  Nodes (entities):  {G.number_of_nodes()}")
print(f"  Edges (relations): {G.number_of_edges()}")

# Relation type distribution
relation_counts = collections.Counter(
    data["relation"] for _, _, data in G.edges(data=True)
)
print(f"\nRelation type distribution:")
for rel, count in relation_counts.most_common():
    print(f"  {rel}: {count}")

---
## Step 4 — Run Comparative Experiment

Runs the three evaluation questions under both conditions.

**Condition A (LLM-only):** raw text → LLM → response  
**Condition B (KG+LLM):** triples subgraph → LLM → response

**Questions:**
- P1 (intra-document): What are the dominant trends in this collection?
- P2 (intra-document): What relations between elements characterize these trends?
- P3 (inter-document): What trends dominated London AW25?

In [ ]:
!python src/04-query.py

---
## Step 5 — Inspect Results

Compare responses from both conditions side by side.

In [ ]:
# Compare Condition A vs B for Erdem P1
def read_result(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

brand = "2025-lfw-erdem-review"
question = "P1"

result_a = read_result(f"results/condition_a/{brand}_{question}.txt")
result_b = read_result(f"results/condition_b/{brand}_{question}.txt")

print("=" * 60)
print("CONDITION A — LLM-only")
print("=" * 60)
print(result_a)

print()
print("=" * 60)
print("CONDITION B — KG+LLM")
print("=" * 60)
print(result_b)

In [ ]:
# Compare Condition A vs B for P3 (inter-document)
result_a_p3 = read_result("results/condition_a/corpus_lfw_aw25_P3.txt")
result_b_p3 = read_result("results/condition_b/corpus_lfw_aw25_P3.txt")

print("=" * 60)
print("P3 — CONDITION A — LLM-only")
print("=" * 60)
print(result_a_p3)

print()
print("=" * 60)
print("P3 — CONDITION B — KG+LLM")
print("=" * 60)
print(result_b_p3)

---
## Conclusion

All results presented in this notebook can be reproduced by anyone with access to:
1. This repository: https://github.com/diegofelixlima/beyond-frequency
2. An OpenAI API key
3. Google Colab (free tier is sufficient)

The notebook contains data, code, and results in a single versioned document,  
in accordance with Open Science principles.

**Evaluation of results** (interpretability and groundedness) is documented  
in the article: `doc/article.tex`